# Molecular dynamics of an explicitly solvated small protein

For the simulations in this notebook, we will make use of the crystal structure of the Villin Headpiece subdomain [1YRF](https://www.rcsb.org/structure/1YRF). This is a relatively small protein (35 residues) consisting of three alpha helices. This is a prototypical fast folding protein, see [10.1016/j.jmb.2006.03.034](https://doi.org/10.1016/j.jmb.2006.03.034) and [10.1073/pnas.0502495102](https://doi.org/10.1073/pnas.0502495102), and it is therefore a popular benchmark for protein folding molecular dynamics simulations, e.g. [10.1073/pnas.1800690115](https://doi.org/10.1073/pnas.1800690115) (force field accuracy) and [10.1109/SC.2014.9](https://doi.org/10.1109/SC.2014.9) (computational performance).

If you want to run a longer simulation, it may be useful to submit it as a non-interactive job,
as explained in [../03_noninteractive/01_noninteractive_notebook_on_hpc.ipynb](../03_noninteractive/01_noninteractive_notebook_on_hpc.ipynb).
The job scripts provided there only load OpenMM and Jupyter, not Matplotlib, Pandas, MDTraj or NGLView.
You should therefore copy the code needed to set up and run the simulation into a separate notebook, without the plotting and visualization, and increase the number of steps there.

In [ ]:
%matplotlib widget
import shutil
from sys import stdout

import matplotlib.pyplot as plt
import mdtraj
import nglview
import openmm as mm
import pandas
from openmm import app, unit

The PDB file of 1YRF contains more information than we need:

- Some residues were resolved in two alternative conformations, called alternate locations.
  The atoms of these residues appear twice in the file, with an extra `A` or `B` just after the atom name.
  OpenMM can read such a file, but it silently keeps only the first alternative.
- Besides the protein, the crystal structure also contains water molecules, a sulfate ion and an acetate ion.
  These are stored as `HETATM` records instead of `ATOM` records.
  The force field used below has no parameters for sulfate and acetate,
  and the crystal waters are not needed because the protein is solvated later on.

The following function splits the PDB file into one file per alternate location, keeping only the `ATOM` records.
This way, the choice of conformation is explicit, and each of the resulting files is suitable for starting an OpenMM simulation.

In [ ]:
def split_pdb(fn_pdb):
    pos = 16
    groups = {}
    counter = 0
    with open(fn_pdb) as f:
        for line in f:
            if line.startswith("ATOM"):
                state = line[pos]
                line = line[:pos] + " " + line[pos + 1 :]
                groups.setdefault(state, []).append((counter, line))
                counter += 1
    print("Group keys:", groups.keys())
    for key, lines_group in groups.items():
        if key == " ":
            continue
        lines_both = lines_group + groups[" "]
        lines_both.sort()
        new_fn_pdb = f"{fn_pdb[:-4]}_{key.lower()}.pdb"
        with open(new_fn_pdb, "w") as f:
            for _counter, line in lines_both:
                f.write(line)
        print("Written", new_fn_pdb)


split_pdb("1yrf.pdb")

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**
> Inspect the contents of the files `1yrf_a.pdb` and `1yrf_b.pdb`. Verify for at least one atom that these files contain the correct information taken from `1yrf.pdb`.

The energy minimization below is monitored with a minimization reporter, as explained in the first notebook of this tutorial ([../01_first_steps/01_water.ipynb](../01_first_steps/01_water.ipynb)).
The reporter below is a bit more elaborate, because the minimization of a solvated protein is more complicated than that of a single water molecule:

- The system contains constraints (`constraints=HBonds` below), but the L-BFGS algorithm cannot handle constraints.
  The minimizer therefore replaces them by harmonic restraints and minimizes the sum of the potential energy and the restraint energy.
  If the constraints are not satisfied well enough afterwards, the restraints are made stronger and the minimization is repeated.
  The reporter stores the `restraint strength`, to see when this happens.

- Every such repetition starts counting iterations from zero again, and `maxIterations` applies to each repetition separately.
  The reporter therefore keeps its own counter to obtain a continuous iteration number.

In [ ]:
class CSVMinimizationReporter(mm.MinimizationReporter):
    """Write the energy after every minimization iteration to a CSV file."""

    def __init__(self, filename):
        super().__init__()
        self.file = open(filename, "w")
        self.file.write("Iteration,Energy (kJ/mole),Restraint strength\n")
        self.counter = 0

    def report(self, iteration, x, grad, args):
        strength = args["restraint strength"]
        self.file.write(f"{self.counter},{args['system energy']},{strength}\n")
        self.counter += 1
        # Returning True would stop the minimization.
        return False

    def close(self):
        self.file.close()

In [ ]:
pdb = app.PDBFile("1yrf_a.pdb")
modeller = app.Modeller(pdb.topology, pdb.positions)
forcefield = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
modeller.addHydrogens(forcefield)
modeller.addSolvent(forcefield, model="tip3p", padding=1 * unit.nanometer)
system = forcefield.createSystem(modeller.topology, nonbondedMethod=app.PME, constraints=app.HBonds)
temperature = 300 * unit.kelvin
pressure = 1 * unit.bar
integrator = mm.LangevinMiddleIntegrator(temperature, 1 / unit.picosecond, 2 * unit.femtoseconds)
system.addForce(mm.MonteCarloBarostat(pressure, temperature))
simulation = app.Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)
reporter = CSVMinimizationReporter("minimization.csv")
simulation.minimizeEnergy(maxIterations=100, reporter=reporter)
reporter.close()
positions = simulation.context.getState(getPositions=True).getPositions()
with open("init.pdb", "w") as f:
    app.PDBFile.writeFile(simulation.topology, positions, f)

The progress of the minimization can be plotted in the same way as in the first notebook.
The dashed vertical lines mark the iterations where the restraint strength was increased.

In [ ]:
df_min = pandas.read_csv("minimization.csv")
plt.close("minimization")
fig, ax = plt.subplots(num="minimization")
df_min.plot(kind="line", x="Iteration", y="Energy (kJ/mole)", ax=ax)
# Mark the iterations where the restraint strength increases.
for iteration in df_min["Iteration"][df_min["Restraint strength"].diff() > 0]:
    ax.axvline(iteration, color="gray", linestyle="--")

Within each repetition, the energy decreases rapidly at first, and then more slowly.
Note that this is the potential energy of the system, without the restraint energy.
It jumps up when the restraints become stronger, because the geometry is then adjusted to satisfy the constraints more precisely.
This is why the final energy is higher than the lowest energy reached during the first repetition, which was obtained with weak restraints.
The minimization is limited to 100 iterations per repetition: the aim is only to remove the worst clashes of atoms in the initial structure, not to find the exact minimum.
In the other notebooks of this tutorial, the minimization is not monitored, because its only purpose is to obtain a reasonable starting point for the molecular dynamics.

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Which changes to the input structure can be made, for which the force field can still be applied? Try the following:
>
> - Remove an atom from the PDB file.
> - Remove an entire residue from the PDB file.
> - Use an engineered form of the Villin Headpiece, e.g. the Lys24Nle mutant ([1WY3](https://www.rcsb.org/structure/1WY3)) or the Lys24Nle/Lys29Nle double mutant ([2F4K](https://www.rcsb.org/structure/2F4K)).
>   Note that norleucine (Nle) is not a standard amino acid: its atoms are stored as `HETATM` records, which are silently discarded by `split_pdb`.
>   Always check which residues end up in the topology.
>
> Finally, restore the input to the original one and rerun the code cell that sets up and minimizes the system, to have a good starting point for the next cell.
>
> Point mutations between standard amino acids can be introduced with PDBFixer, which is used in [../05_analysis/03_alignment_principal_component_analysis.ipynb](../05_analysis/03_alignment_principal_component_analysis.ipynb).
> Nonstandard residues, such as Nle, also require additional force-field parameters, which is beyond the scope of this tutorial.

The next cell runs a short molecular dynamics simulation of 30000 steps of $2\,\mathrm{fs}$, i.e. $60\,\mathrm{ps}$.
Every 10 steps, the atomic positions are written to `traj.dcd` and a few scalar properties to `scalars.csv`.
The progress is printed on screen every 100 steps.

As in [../02_alanine_dipeptide/01_force_fields.ipynb](../02_alanine_dipeptide/01_force_fields.ipynb), no initial velocities are assigned:
the atoms start at rest after the energy minimization, and the Langevin thermostat gradually heats the system up to $300\,\mathrm{K}$.

In [ ]:
simulation.reporters = []
simulation.reporters.append(app.DCDReporter("traj.dcd", 10))
simulation.reporters.append(
    app.StateDataReporter(stdout, 100, step=True, temperature=True, speed=True, elapsedTime=True)
)
simulation.reporters.append(
    app.StateDataReporter(
        "scalars.csv",
        10,
        step=True,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
    )
)
simulation.step(30000)

The potential energy as a function of time reveals the warm-up phase at the start of the simulation.
During the first fraction of a picosecond, the potential energy still decreases,
because the minimization was stopped after a limited number of iterations and the (still cold) system continues to relax.
It then increases while the thermostat heats up the system.
After a few picoseconds, it fluctuates around a constant value.

In [ ]:
df = pandas.read_csv("scalars.csv")
plt.close("ener")
fig, ax = plt.subplots(num="ener")
df.plot(kind="line", x="Time (ps)", y="Potential Energy (kJ/mole)", ax=ax);

The trajectory can be visualized in the same way as in the previous notebooks.

In [ ]:
traj = mdtraj.load("traj.dcd", top="init.pdb")
view = nglview.show_mdtraj(traj)
view

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Make an estimate of the computational cost (wall time) on your current hardware to run a sufficiently long simulation to observe spontaneously a protein folding event for this fast folder. The required simulation time is approximately $4\,\mu\mathrm{s}$. Would such a calculation be feasible?

You will need the output files of this simulation for the notebooks in `../05_analysis`. Copy over the files `init.pdb`, `scalars.csv` and `traj.dcd` to that directory. This can be done with the following code, or with any file manager that comes with the operating system.

In [ ]:
shutil.copy("init.pdb", "../05_analysis")
shutil.copy("scalars.csv", "../05_analysis")
shutil.copy("traj.dcd", "../05_analysis")